# SingBERT Training on SMSSpamCollection

This notebook mirrors the main classifier training flow, but trains on `datasets/SMSSpamCollection_train_subset` and evaluates on `datasets/SMSSpamCollection_ablation_test`.


## Environment Setup


In [ ]:
# Install only the extra packages needed in Colab.
# Leave Colab's torch / torchvision versions alone.
# Reinstalling one without the other can break CUDA support.
%pip install -q transformers datasets accelerate


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


## Project Configuration


In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "datasets"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_OUTPUT_DIR = ARTIFACT_DIR / "smsspam_singbert_classifier"
METRICS_OUTPUT_DIR = ARTIFACT_DIR / "smsspam_singbert_metrics"

TRAIN_INPUT_PATH = DATA_DIR / "SMSSpamCollection_train_subset"
TEST_INPUT_PATH = DATA_DIR / "SMSSpamCollection_ablation_test"

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

MODEL_CHECKPOINT = "zanelim/singbert"

RANDOM_SEED = 42
VALIDATION_SIZE = 0.15

# Keep the same classifier hyperparameters as the main SingBERT notebook.
MAX_LENGTH = 128
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 160)

for data_path in [TRAIN_INPUT_PATH, TEST_INPUT_PATH]:
    if not data_path.exists():
        raise FileNotFoundError(f"Missing input dataset: {data_path}")

print(f"Training dataset: {TRAIN_INPUT_PATH}")
print(f"Ablation test dataset: {TEST_INPUT_PATH}")


## Modeling Dataset Preparation


In [ ]:
# Load the SMS spam train and test files and map labels to scam / not_scam.
def load_sms_dataframe(data_path):
    rows = []
    with open(data_path, "r", encoding="utf-8", errors="replace") as fp:
        for line_number, raw_line in enumerate(fp, start=1):
            line = raw_line.rstrip("\n")
            if not line:
                continue
            if "\t" not in line:
                raise ValueError(f"Line {line_number} is missing a tab separator.")
            raw_label, text = line.split("\t", 1)
            rows.append({"raw_label": raw_label.strip(), TEXT_COLUMN: text.strip()})
    return pd.DataFrame(rows)

train_source_df = load_sms_dataframe(TRAIN_INPUT_PATH)
test_df = load_sms_dataframe(TEST_INPUT_PATH)

label_map = {"ham": "not_scam", "spam": "scam"}
for df in [train_source_df, test_df]:
    df[LABEL_COLUMN] = df["raw_label"].map(label_map)

if train_source_df[LABEL_COLUMN].isna().any() or test_df[LABEL_COLUMN].isna().any():
    bad_labels = sorted(
        set(train_source_df.loc[train_source_df[LABEL_COLUMN].isna(), "raw_label"].unique())
        | set(test_df.loc[test_df[LABEL_COLUMN].isna(), "raw_label"].unique())
    )
    raise ValueError(f"Unexpected labels in SMSSpamCollection: {bad_labels}")

for df in [train_source_df, test_df]:
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()
    df.drop_duplicates(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)
    df.reset_index(drop=True, inplace=True)

label_names = sorted(train_source_df[LABEL_COLUMN].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}
train_source_df["labels"] = train_source_df[LABEL_COLUMN].map(label2id)
test_df["labels"] = test_df[LABEL_COLUMN].map(label2id)

display(train_source_df.head())
print("Training source counts:")
print(train_source_df[LABEL_COLUMN].value_counts().sort_index().to_string())
print()
print("Held-out ablation test counts:")
print(test_df[LABEL_COLUMN].value_counts().sort_index().to_string())


In [ ]:
# Split the training subset into train and validation, and keep the ablation set fixed as test
train_df, validation_df = train_test_split(
    train_source_df,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_source_df["labels"],
)

split_frames = {"train": train_df, "validation": validation_df, "test": test_df}
for split_name, split_df in split_frames.items():
    print(f"{split_name:>10}: {len(split_df):4d} rows")
    print(split_df[LABEL_COLUMN].value_counts().sort_index().to_string())
    print()

dataset_dict = DatasetDict(
    {
        split_name: Dataset.from_pandas(split_df.reset_index(drop=True), preserve_index=False)
        for split_name, split_df in split_frames.items()
    }
)


## Tokenization Setup


In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)

tensor_columns = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized_datasets["train"].column_names:
    tensor_columns.append("token_type_ids")

tokenized_datasets.set_format(type="torch", columns=tensor_columns)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

label_mapping_path = METRICS_OUTPUT_DIR / "label_mapping.json"
with open(label_mapping_path, "w", encoding="utf-8") as fp:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
        },
        fp,
        indent=2,
    )

print(f"Saved label mapping to: {label_mapping_path}")


## Training Pipeline


In [ ]:
def calculate_classification_metrics(true_ids, predicted_ids):
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy_score(true_ids, predicted_ids),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return calculate_classification_metrics(labels, predictions)


In [ ]:
# Load the classification model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

training_args_kwargs = dict(
    output_dir=str(MODEL_OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    seed=RANDOM_SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

if "eval_strategy" in TrainingArguments.__dataclass_fields__:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))


### Evaluation Helpers


In [ ]:
# Convert NumPy scalars to plain Python values
def to_python_scalars(metrics_dict):
    cleaned = {}
    for key, value in metrics_dict.items():
        if isinstance(value, (np.floating, np.integer)):
            cleaned[key] = value.item()
        else:
            cleaned[key] = value
    return cleaned

def evaluate_split(split_name, dataset_split):
    prediction_output = trainer.predict(dataset_split, metric_key_prefix=split_name)
    predicted_ids = prediction_output.predictions.argmax(axis=-1)
    true_ids = prediction_output.label_ids

    split_metrics = to_python_scalars(
        {
            "loss": prediction_output.metrics.get(f"{split_name}_loss"),
            **calculate_classification_metrics(true_ids, predicted_ids),
        }
    )

    report = classification_report(
        true_ids,
        predicted_ids,
        labels=list(range(len(label2id))),
        target_names=[id2label[idx] for idx in range(len(label2id))],
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_path = METRICS_OUTPUT_DIR / f"{split_name}_classification_report.csv"
    report_df.to_csv(report_path)

    return split_metrics, report_df


In [ ]:
# Evaluate all dataset splits
validation_metrics, validation_report_df = evaluate_split("validation", tokenized_datasets["validation"])
train_metrics, train_report_df = evaluate_split("train", tokenized_datasets["train"])
test_metrics, test_report_df = evaluate_split("test", tokenized_datasets["test"])

summary_metrics = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "train_input_path": str(TRAIN_INPUT_PATH),
    "test_input_path": str(TEST_INPUT_PATH),
    "output_model_dir": str(MODEL_OUTPUT_DIR),
    "split_sizes": {split_name: len(split_df) for split_name, split_df in split_frames.items()},
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "train_runtime": to_python_scalars(train_result.metrics),
    "validation": validation_metrics,
    "train": train_metrics,
    "test": test_metrics,
}

summary_metrics_path = METRICS_OUTPUT_DIR / "summary_metrics.json"
with open(summary_metrics_path, "w", encoding="utf-8") as fp:
    json.dump(summary_metrics, fp, indent=2)

trainer_log_path = METRICS_OUTPUT_DIR / "trainer_log_history.json"
with open(trainer_log_path, "w", encoding="utf-8") as fp:
    json.dump([to_python_scalars(item) for item in trainer.state.log_history], fp, indent=2)

metrics_overview = pd.DataFrame([train_metrics, validation_metrics, test_metrics], index=["train", "validation", "test"])
display(metrics_overview)
display(test_report_df)

print(f"Saved fine-tuned model to: {MODEL_OUTPUT_DIR}")
print(f"Saved metric summary to: {summary_metrics_path}")
print(f"Saved trainer history to: {trainer_log_path}")
print(f"Saved per-class reports to: {METRICS_OUTPUT_DIR}")
